<a href="https://colab.research.google.com/github/HannanSeyfi/unlearning-thesis/blob/main/Week%205/notebooks/week5_retain_regularized_unlearning_resumable.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5: Resumable Retain-Regularized Unlearning on Qwen 0.5B

This notebook is a resumable version of `week5_retain_regularized_unlearning.ipynb`. It starts from the same strict-scored Week 3.5 LoRA adapter, runs the same retain-regularized sweep, and saves progress to Google Drive after every candidate epoch so the sweep can continue later on another Colab or host.

The original Week 5 notebook is not modified by this notebook.


## 1. Install Dependencies

In [1]:
%pip uninstall -q -y bitsandbytes
%pip install -q -U "transformers==4.48.3" "accelerate==1.3.0" "peft==0.19.1" "datasets==3.2.0" sentencepiece "pandas==2.2.3"
%pip install -q --no-deps "bitsandbytes==0.49.2"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.6/336.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
gcsfs 2025.3.0 r

## 2. Mount Drive and Set Paths

In [2]:
from pathlib import Path
import json
from google.colab import drive

drive.mount('/content/drive')
THESIS_DIR = Path('/content/drive/MyDrive/Thesis')
WEEK35_DIR = THESIS_DIR / 'Week 3.5'
DATA_DIR = WEEK35_DIR / 'data' / 'synthetic_facts_v1'
CONTROL_DIR = WEEK35_DIR / 'data' / 'general_controls_v1'
SOURCE_ADAPTER_DIR = WEEK35_DIR / 'results' / 'qwen05_high_accuracy_baseline' / 'adapter'
STRICT_WEEK35_METRICS_PATH = WEEK35_DIR / 'results' / 'qwen05_high_accuracy_baseline' / 'metrics.json'

RUN_DIR = THESIS_DIR / 'Week 5' / 'results' / 'retain_regularized_unlearning_resumable_v1'
ADAPTER_DIR = RUN_DIR / 'best_retain_regularized_adapter'
CANDIDATE_ADAPTER_DIR = RUN_DIR / 'candidate_adapters'
OUTPUT_DIR = RUN_DIR / 'results'
RESUME_DIR = RUN_DIR / 'resume_state'
EPOCH_CHECKPOINT_DIR = RESUME_DIR / 'epoch_checkpoints'
SELECTION_RESULT_DIR = RESUME_DIR / 'selection_results'
for folder in [ADAPTER_DIR, CANDIDATE_ADAPTER_DIR, OUTPUT_DIR, RESUME_DIR, EPOCH_CHECKPOINT_DIR, SELECTION_RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TRAIN_FORGET_PATH = DATA_DIR / 'train_forget.jsonl'
TRAIN_RETAIN_PATH = DATA_DIR / 'train_retain.jsonl'
EVAL_FORGET_PATH = DATA_DIR / 'eval_forget.jsonl'
EVAL_RETAIN_PATH = DATA_DIR / 'eval_retain.jsonl'
GENERAL_CONTROL_PATH = CONTROL_DIR / 'general_control.jsonl'
required_paths = [SOURCE_ADAPTER_DIR, STRICT_WEEK35_METRICS_PATH, TRAIN_FORGET_PATH, TRAIN_RETAIN_PATH, EVAL_FORGET_PATH, EVAL_RETAIN_PATH, GENERAL_CONTROL_PATH]
for path in required_paths:
    assert path.exists(), f'Missing required file or folder: {path}. Run Week 3.5/notebooks/week3_5_train_high_accuracy_baseline_strict.ipynb first.'
strict_week35_metrics = json.loads(STRICT_WEEK35_METRICS_PATH.read_text(encoding='utf-8'))
assert 'strict_scoring' in strict_week35_metrics, 'Week 3.5 metrics do not contain strict_scoring. Run the strict Week 3.5 baseline or strict re-evaluation notebook before Week 5.'
print('Starting strict Week 3.5 adapter:', SOURCE_ADAPTER_DIR)
print('Week 3.5 strict metrics:', STRICT_WEEK35_METRICS_PATH)
print('Week 3.5 run:', strict_week35_metrics.get('run_name'))
print('Outputs:', RUN_DIR)
print('Resume state:', RESUME_DIR)


Mounted at /content/drive
Starting strict Week 3.5 adapter: /content/drive/MyDrive/Thesis/Week 3.5/results/qwen05_high_accuracy_baseline/adapter
Week 3.5 strict metrics: /content/drive/MyDrive/Thesis/Week 3.5/results/qwen05_high_accuracy_baseline/metrics.json
Week 3.5 run: week3_5_qwen05_high_accuracy_baseline
Outputs: /content/drive/MyDrive/Thesis/Week 5/results/retain_regularized_unlearning_resumable_v1
Resume state: /content/drive/MyDrive/Thesis/Week 5/results/retain_regularized_unlearning_resumable_v1/resume_state


## 3. Configuration, Sweep Grid, Resume Settings, and Data

Default mode runs a focused 9-candidate sweep. Set `RUN_FULL_GRID = True` to run all 27 combinations. Resume is enabled by default and uses files under `resume_state/`.


In [3]:
import gc
import itertools
import json
import random
import re
import shutil
import unicodedata
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from peft import PeftModel, prepare_model_for_kbit_training
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SEED = 42
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_LENGTH = 192
MAX_EPOCHS = 8
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_GRAD_NORM = 1.0
TARGET_FORGET_MAX = 20.0
MIN_RETAIN_HELDOUT_SELECTION = 85.0
SYNTHETIC_SYSTEM_PROMPT = 'You answer questions about fictional synthetic people using the provided learned facts.'
GENERAL_SYSTEM_PROMPT = 'Answer the question concisely. Return only the requested answer without explanation.'

RETAIN_WEIGHTS = [1.0, 2.0, 4.0]
KL_WEIGHTS = [0.1, 0.5, 1.0]
LEARNING_RATES = [1e-5, 2e-5, 5e-5]
RUN_FULL_GRID = False
RESUME_SWEEP = True
RESET_RESUME_STATE = False
REUSE_EXISTING_BEFORE_EVAL = True

FOCUSED_SWEEP = [
    {'learning_rate': 1e-5, 'retain_weight': 2.0, 'kl_weight': 0.5, 'label': 'low_lr_balanced'},
    {'learning_rate': 2e-5, 'retain_weight': 2.0, 'kl_weight': 0.5, 'label': 'mid_lr_balanced'},
    {'learning_rate': 5e-5, 'retain_weight': 2.0, 'kl_weight': 0.5, 'label': 'high_lr_balanced'},
    {'learning_rate': 2e-5, 'retain_weight': 1.0, 'kl_weight': 0.5, 'label': 'lower_retain'},
    {'learning_rate': 2e-5, 'retain_weight': 4.0, 'kl_weight': 0.5, 'label': 'higher_retain'},
    {'learning_rate': 2e-5, 'retain_weight': 2.0, 'kl_weight': 0.1, 'label': 'lower_kl'},
    {'learning_rate': 2e-5, 'retain_weight': 2.0, 'kl_weight': 1.0, 'label': 'higher_kl'},
    {'learning_rate': 1e-5, 'retain_weight': 4.0, 'kl_weight': 1.0, 'label': 'most_preserving'},
    {'learning_rate': 5e-5, 'retain_weight': 1.0, 'kl_weight': 0.1, 'label': 'most_aggressive'},
]

if RUN_FULL_GRID:
    SWEEP_CONFIGS = [
        {'learning_rate': lr, 'retain_weight': rw, 'kl_weight': kw, 'label': f'lr{lr:g}_retain{rw:g}_kl{kw:g}'}
        for lr, rw, kw in itertools.product(LEARNING_RATES, RETAIN_WEIGHTS, KL_WEIGHTS)
    ]
else:
    SWEEP_CONFIGS = FOCUSED_SWEEP

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
torch.cuda.manual_seed_all(SEED)
print('GPU:', torch.cuda.get_device_name(0))
print('Sweep candidates:', len(SWEEP_CONFIGS))
print('Resume sweep:', RESUME_SWEEP)
print('Reuse existing before-evaluation files:', REUSE_EXISTING_BEFORE_EVAL)


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]

train_forget = read_jsonl(TRAIN_FORGET_PATH)
train_retain = read_jsonl(TRAIN_RETAIN_PATH)
eval_forget = read_jsonl(EVAL_FORGET_PATH)
eval_retain = read_jsonl(EVAL_RETAIN_PATH)
general_controls = read_jsonl(GENERAL_CONTROL_PATH)
assert len(train_forget) == 100 and len(train_retain) == 400
assert len(eval_forget) == 300 and len(eval_retain) == 1200
assert len(general_controls) == 50
original_train_prompt_keys = {
    (row['entity_id'], row['fact_type'], row['prompt'].strip().lower())
    for row in train_forget + train_retain
}

def is_seen_prompt(row):
    return (row.get('entity_id'), row.get('fact_type'), row['prompt'].strip().lower()) in original_train_prompt_keys

heldout_forget = [row for row in eval_forget if not is_seen_prompt(row)]
heldout_retain = [row for row in eval_retain if not is_seen_prompt(row)]
selection_rng = random.Random(SEED + 500)
forget_selection = selection_rng.sample(heldout_forget, 80)
retain_selection = selection_rng.sample(heldout_retain, 160)
selection_ids = {row['example_id'] for row in forget_selection + retain_selection}
final_forget_excluding_selection = [row for row in eval_forget if row.get('example_id') not in selection_ids]
final_retain_excluding_selection = [row for row in eval_retain if row.get('example_id') not in selection_ids]

print('Forget train/eval/heldout-selection:', len(train_forget), len(eval_forget), len(forget_selection))
print('Retain train/eval/heldout-selection:', len(train_retain), len(eval_retain), len(retain_selection))
print('Final excluding selection:', len(final_forget_excluding_selection), len(final_retain_excluding_selection))
print('General controls:', len(general_controls))

GPU: Tesla T4
Sweep candidates: 9
Resume sweep: True
Reuse existing before-evaluation files: True
Forget train/eval/heldout-selection: 100 300 80
Retain train/eval/heldout-selection: 400 1200 160
Final excluding selection: 220 1040
General controls: 50


## 4. Model and Evaluation Helpers

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'


def load_base_model():
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=quantization, device_map='auto'
    )


def normalize_text(text):
    text = unicodedata.normalize('NFKD', str(text))
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r'\s+', ' ', text.strip().lower())
    return text.strip(" .,!?:;\"'")


def contains_value(generated, expected):
    generated, expected = normalize_text(generated), normalize_text(expected)
    return bool(expected and re.search(rf'(?<!\w){re.escape(expected)}(?!\w)', generated))


@torch.inference_mode()
def generate_answer(model, prompt, general=False):
    messages = [
        {'role': 'system', 'content': GENERAL_SYSTEM_PROMPT if general else SYNTHETIC_SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=16,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()


def evaluate(model, records, split, stage, general=False, progress_every=100):
    rows = []
    for index, row in enumerate(records, 1):
        expected = str(row.get('fact_value', row.get('expected_value', '')))
        answer = generate_answer(model, row['prompt'], general=general)
        example_id = row.get('example_id')
        rows.append({
            'model_stage': stage,
            'eval_split': split,
            'used_for_checkpoint_selection': example_id in selection_ids if example_id is not None else False,
            'prompt_seen_in_original_training': is_seen_prompt(row) if not general else False,
            'example_id': example_id,
            'entity_id': row.get('entity_id'),
            'category': row.get('fact_type', row.get('category')),
            'prompt': row['prompt'],
            'expected_value': expected,
            'generated_answer': answer,
            'exact_match': normalize_text(answer) == normalize_text(expected),
            'contains_value': contains_value(answer, expected),
        })
        if progress_every and index % progress_every == 0:
            print(f'{stage} {split}: {index}/{len(records)}')
    return pd.DataFrame(rows)


def percentage(frame):
    return float(100 * frame['contains_value'].mean())


def prompt_subset_percentage(frame, seen):
    subset = frame[frame['prompt_seen_in_original_training'] == seen]
    return float(100 * subset['contains_value'].mean())


def excluding_selection_percentage(frame):
    subset = frame[~frame['used_for_checkpoint_selection']]
    return float(100 * subset['contains_value'].mean())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 5. Evaluate the Learned Model Before Week 5 Unlearning

In [5]:
before_forget_path = OUTPUT_DIR / 'before_forget_results.csv'
before_retain_path = OUTPUT_DIR / 'before_retain_results.csv'
before_general_path = OUTPUT_DIR / 'before_general_results.csv'

if REUSE_EXISTING_BEFORE_EVAL and before_forget_path.exists() and before_retain_path.exists() and before_general_path.exists():
    before_forget_df = pd.read_csv(before_forget_path)
    before_retain_df = pd.read_csv(before_retain_path)
    before_general_df = pd.read_csv(before_general_path)
    print('Loaded existing before-unlearning evaluation files.')
else:
    model = load_base_model()
    model = PeftModel.from_pretrained(model, SOURCE_ADAPTER_DIR)
    model.eval()
    before_forget_df = evaluate(model, eval_forget, 'forget', 'before_unlearning')
    before_retain_df = evaluate(model, eval_retain, 'retain', 'before_unlearning')
    before_general_df = evaluate(model, general_controls, 'general', 'before_unlearning', general=True, progress_every=0)
    before_forget_df.to_csv(before_forget_path, index=False)
    before_retain_df.to_csv(before_retain_path, index=False)
    before_general_df.to_csv(before_general_path, index=False)
    del model
    gc.collect()
    torch.cuda.empty_cache()

print('Before forget:', f'{percentage(before_forget_df):.2f}%')
print('Before retain:', f'{percentage(before_retain_df):.2f}%')
print('Before general:', f'{percentage(before_general_df):.2f}%')


Loaded existing before-unlearning evaluation files.
Before forget: 95.00%
Before retain: 94.58%
Before general: 56.00%


## 6. Build Training Batches and KL Regularizer

In [6]:
def encode_record(row):
    messages = [
        {'role': 'system', 'content': SYNTHETIC_SYSTEM_PROMPT},
        {'role': 'user', 'content': row['prompt']},
        {'role': 'assistant', 'content': str(row['fact_value'])},
    ]
    prompt_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    full = tokenizer(full_text, truncation=True, max_length=MAX_LENGTH, padding='max_length')
    prompt = tokenizer(prompt_text, truncation=True, max_length=MAX_LENGTH, padding=False)
    labels = full['input_ids'].copy()
    prompt_len = min(len(prompt['input_ids']), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    labels = [label if mask else -100 for label, mask in zip(labels, full['attention_mask'])]
    return {
        'input_ids': torch.tensor(full['input_ids']),
        'attention_mask': torch.tensor(full['attention_mask']),
        'labels': torch.tensor(labels),
    }


class EncodedDataset(Dataset):
    def __init__(self, rows):
        self.rows = [encode_record(row) for row in rows]
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        return self.rows[index]


forget_dataset = EncodedDataset(train_forget)
retain_dataset = EncodedDataset(train_retain)


def make_loaders(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    forget_loader = DataLoader(forget_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)
    retain_loader = DataLoader(retain_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)
    return forget_loader, retain_loader


def masked_token_kl(student_logits, teacher_logits, labels):
    # CausalLM loss predicts labels[:, 1:] from logits[:, :-1, :]. Match that shift here.
    shifted_student_logits = student_logits[:, :-1, :]
    shifted_teacher_logits = teacher_logits[:, :-1, :]
    shifted_labels = labels[:, 1:]
    mask = shifted_labels.ne(-100)
    if not bool(mask.any()):
        return student_logits.new_tensor(0.0)
    student_log_probs = F.log_softmax(shifted_student_logits.float(), dim=-1)
    teacher_probs = F.softmax(shifted_teacher_logits.float(), dim=-1)
    token_kl = F.kl_div(student_log_probs, teacher_probs, reduction='none').sum(dim=-1)
    return (token_kl * mask.float()).sum() / mask.float().sum().clamp_min(1.0)

## 7. Run Resumable Retain-Regularized Sweep

Progress is saved after every candidate epoch. If the runtime disconnects, rerun the setup cells and this cell will skip completed candidates or resume the incomplete candidate from its latest epoch checkpoint.

The notebook saves adapter checkpoints, optimizer state, selection CSVs, `sweep_history.csv`, `candidate_best_summary.csv`, and `resume_state/global_state.json` to Google Drive.


In [8]:
def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding='utf-8')


def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding='utf-8'))


def candidate_id_from_config(config_index, config):
    return f"c{config_index:02d}_{config['label']}"


def candidate_resume_dir(candidate_id):
    return EPOCH_CHECKPOINT_DIR / candidate_id


def latest_checkpoint_info(candidate_id):
    return read_json(candidate_resume_dir(candidate_id) / 'latest.json')


def move_optimizer_state_to_device(optimizer, device):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)


def save_epoch_checkpoint(model, optimizer, candidate_id, epoch, row):
    epoch_dir = candidate_resume_dir(candidate_id) / f'epoch_{epoch:02d}'
    adapter_dir = epoch_dir / 'adapter'
    adapter_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    training_state_path = epoch_dir / 'training_state.pt'
    torch.save({
        'optimizer_state_dict': optimizer.state_dict(),
        'completed_epoch': epoch,
        'candidate_id': candidate_id,
        'row': row,
        'python_random_state': random.getstate(),
        'numpy_random_state': np.random.get_state(),
        'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state_all': torch.cuda.get_rng_state_all(),
    }, training_state_path)
    latest = {
        'candidate_id': candidate_id,
        'latest_epoch': epoch,
        'adapter_dir': str(adapter_dir),
        'training_state_path': str(training_state_path),
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    write_json(candidate_resume_dir(candidate_id) / 'latest.json', latest)
    return latest


def save_progress_tables(sweep_history, candidate_summaries):
    sweep_history_df = pd.DataFrame(sweep_history)
    sweep_history_df.to_csv(OUTPUT_DIR / 'sweep_history.csv', index=False)
    if candidate_summaries:
        candidate_summary_df = pd.DataFrame(candidate_summaries)
        candidate_summary_df = candidate_summary_df.drop_duplicates('candidate_id', keep='last')
        candidate_summary_df = candidate_summary_df.sort_values('selection_score', ascending=False)
    else:
        candidate_summary_df = pd.DataFrame()
    candidate_summary_df.to_csv(OUTPUT_DIR / 'candidate_best_summary.csv', index=False)
    return sweep_history_df, candidate_summary_df


if RESET_RESUME_STATE:
    assert not RESUME_SWEEP, 'Set RESUME_SWEEP = False when RESET_RESUME_STATE = True.'
    if RESUME_DIR.exists():
        shutil.rmtree(RESUME_DIR)
    if CANDIDATE_ADAPTER_DIR.exists():
        shutil.rmtree(CANDIDATE_ADAPTER_DIR)
    for folder in [RESUME_DIR, EPOCH_CHECKPOINT_DIR, SELECTION_RESULT_DIR, CANDIDATE_ADAPTER_DIR]:
        folder.mkdir(parents=True, exist_ok=True)

existing_history_path = OUTPUT_DIR / 'sweep_history.csv'
existing_summary_path = OUTPUT_DIR / 'candidate_best_summary.csv'
if RESUME_SWEEP and existing_history_path.exists():
    sweep_history = pd.read_csv(existing_history_path).to_dict('records')
else:
    sweep_history = []
if RESUME_SWEEP and existing_summary_path.exists():
    existing_summary_df = pd.read_csv(existing_summary_path)
    candidate_summaries = existing_summary_df.to_dict('records') if not existing_summary_df.empty else []
else:
    candidate_summaries = []
completed_candidate_ids = {row['candidate_id'] for row in candidate_summaries if 'candidate_id' in row}
print('Existing epoch rows:', len(sweep_history))
print('Completed candidates:', sorted(completed_candidate_ids))

teacher_base = load_base_model()
teacher_model = PeftModel.from_pretrained(teacher_base, SOURCE_ADAPTER_DIR)
teacher_model.eval()
for parameter in teacher_model.parameters():
    parameter.requires_grad_(False)

for config_index, config in enumerate(SWEEP_CONFIGS, 1):
    config_id = candidate_id_from_config(config_index, config)
    candidate_dir = CANDIDATE_ADAPTER_DIR / config_id

    if RESUME_SWEEP and config_id in completed_candidate_ids:
        print(f'Skipping completed candidate {config_id}')
        continue

    print('\n' + '=' * 80)
    print('Candidate:', config_id, config)
    print('=' * 80)

    candidate_rows = [row for row in sweep_history if row.get('candidate_id') == config_id]
    latest = latest_checkpoint_info(config_id) if RESUME_SWEEP else None
    start_epoch = 1
    resume_adapter_dir = SOURCE_ADAPTER_DIR
    resume_training_state = None
    if latest:
        start_epoch = int(latest['latest_epoch']) + 1
        resume_adapter_dir = Path(latest['adapter_dir'])
        resume_training_state = Path(latest['training_state_path'])
        print(f'Resuming {config_id} from epoch {start_epoch} using {resume_adapter_dir}')
        if start_epoch > MAX_EPOCHS:
            print(f'{config_id} already has all {MAX_EPOCHS} epochs but was not marked complete. Finalizing from history.')
    else:
        if candidate_dir.exists() and not RESUME_SWEEP:
            shutil.rmtree(candidate_dir)
        candidate_dir.mkdir(parents=True, exist_ok=True)

    base_model = load_base_model()
    base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
    model = PeftModel.from_pretrained(base_model, resume_adapter_dir, is_trainable=True)
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    optimizer = AdamW(trainable, lr=config['learning_rate'])

    if resume_training_state and resume_training_state.exists():
        try:
            state = torch.load(resume_training_state, map_location='cpu', weights_only=False)
        except TypeError:
            state = torch.load(resume_training_state, map_location='cpu')
        optimizer.load_state_dict(state['optimizer_state_dict'])
        move_optimizer_state_to_device(optimizer, model.device)

    candidate_best = {'score': float('-inf'), 'epoch': None, 'row': None}
    for row in candidate_rows:
        if row.get('selection_score', float('-inf')) > candidate_best['score']:
            candidate_best = {'score': row['selection_score'], 'epoch': int(row['epoch']), 'row': row.copy()}

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_forget_loss, epoch_retain_loss, epoch_kl_loss = [], [], []
        forget_loader, retain_loader = make_loaders(SEED + config_index * 1000 + epoch)
        retain_iterator = iter(retain_loader)

        for step, forget_batch in enumerate(forget_loader, 1):
            try:
                retain_batch = next(retain_iterator)
            except StopIteration:
                retain_iterator = iter(retain_loader)
                retain_batch = next(retain_iterator)

            forget_batch = {key: value.to(model.device) for key, value in forget_batch.items()}
            retain_batch = {key: value.to(model.device) for key, value in retain_batch.items()}

            forget_loss = model(**forget_batch).loss
            retain_outputs = model(**retain_batch)
            retain_loss = retain_outputs.loss
            with torch.no_grad():
                teacher_outputs = teacher_model(
                    input_ids=retain_batch['input_ids'],
                    attention_mask=retain_batch['attention_mask'],
                )
            kl_loss = masked_token_kl(retain_outputs.logits, teacher_outputs.logits, retain_batch['labels'])

            objective = (
                -forget_loss
                + config['retain_weight'] * retain_loss
                + config['kl_weight'] * kl_loss
            ) / GRADIENT_ACCUMULATION_STEPS
            objective.backward()

            epoch_forget_loss.append(float(forget_loss.detach()))
            epoch_retain_loss.append(float(retain_loss.detach()))
            epoch_kl_loss.append(float(kl_loss.detach()))

            if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(forget_loader):
                torch.nn.utils.clip_grad_norm_(trainable, MAX_GRAD_NORM)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

        model.config.use_cache = True
        model.eval()
        selection_forget_df = evaluate(model, forget_selection, 'forget_heldout_selection', f'{config_id}_epoch_{epoch}', progress_every=0)
        selection_retain_df = evaluate(model, retain_selection, 'retain_heldout_selection', f'{config_id}_epoch_{epoch}', progress_every=0)
        selection_candidate_dir = SELECTION_RESULT_DIR / config_id
        selection_candidate_dir.mkdir(parents=True, exist_ok=True)
        selection_forget_df.to_csv(selection_candidate_dir / f'epoch_{epoch:02d}_forget_selection.csv', index=False)
        selection_retain_df.to_csv(selection_candidate_dir / f'epoch_{epoch:02d}_retain_selection.csv', index=False)

        forget_pct = percentage(selection_forget_df)
        retain_pct = percentage(selection_retain_df)
        eligible = retain_pct >= MIN_RETAIN_HELDOUT_SELECTION
        score = (100.0 - forget_pct) + 0.25 * (retain_pct - MIN_RETAIN_HELDOUT_SELECTION) if eligible else -1000.0 + retain_pct - forget_pct

        row = {
            'candidate_id': config_id,
            'epoch': epoch,
            'learning_rate': config['learning_rate'],
            'retain_weight': config['retain_weight'],
            'kl_weight': config['kl_weight'],
            'mean_forget_loss': float(np.mean(epoch_forget_loss)),
            'mean_retain_loss': float(np.mean(epoch_retain_loss)),
            'mean_kl_loss': float(np.mean(epoch_kl_loss)),
            'forget_heldout_selection_percentage': forget_pct,
            'retain_heldout_selection_percentage': retain_pct,
            'retain_eligible': eligible,
            'selection_score': score,
            'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        }
        sweep_history = [old for old in sweep_history if not (old.get('candidate_id') == config_id and int(old.get('epoch')) == epoch)]
        sweep_history.append(row)
        latest = save_epoch_checkpoint(model, optimizer, config_id, epoch, row)
        print(row)
        print('Saved epoch checkpoint:', latest['adapter_dir'])

        if score > candidate_best['score']:
            candidate_best = {'score': score, 'epoch': epoch, 'row': row.copy()}
            if candidate_dir.exists():
                shutil.rmtree(candidate_dir)
            model.save_pretrained(candidate_dir)
            tokenizer.save_pretrained(candidate_dir)
            print('Updated candidate-best adapter:', candidate_dir)

        sweep_history_df, candidate_summary_df = save_progress_tables(sweep_history, candidate_summaries)
        write_json(RESUME_DIR / 'global_state.json', {
            'updated_at_utc': datetime.now(timezone.utc).isoformat(),
            'active_candidate_id': config_id,
            'latest_epoch': epoch,
            'completed_candidate_ids': sorted(completed_candidate_ids),
            'num_epoch_rows': len(sweep_history),
            'run_dir': str(RUN_DIR),
        })

        model.config.use_cache = False
        if eligible and forget_pct <= TARGET_FORGET_MAX:
            print('Early-stop target reached for this candidate.')
            break

    if candidate_best['row'] is None:
        raise RuntimeError(f'No completed epoch rows were available for {config_id}.')

    candidate_summary = candidate_best['row'].copy()
    candidate_summary['selected_epoch_for_candidate'] = int(candidate_best['epoch'])
    candidate_summary['candidate_adapter_dir'] = str(candidate_dir)
    candidate_summaries = [row for row in candidate_summaries if row.get('candidate_id') != config_id]
    candidate_summaries.append(candidate_summary)
    completed_candidate_ids.add(config_id)
    sweep_history_df, candidate_summary_df = save_progress_tables(sweep_history, candidate_summaries)
    write_json(RESUME_DIR / 'global_state.json', {
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        'active_candidate_id': None,
        'completed_candidate_ids': sorted(completed_candidate_ids),
        'num_epoch_rows': len(sweep_history),
        'num_completed_candidates': len(completed_candidate_ids),
        'run_dir': str(RUN_DIR),
    })

    del model, base_model, optimizer
    gc.collect()
    torch.cuda.empty_cache()

sweep_history_df, candidate_summary_df = save_progress_tables(sweep_history, candidate_summaries)
if candidate_summary_df.empty:
    raise RuntimeError('No completed candidates found. Resume the sweep before final evaluation.')

best_row = candidate_summary_df.sort_values('selection_score', ascending=False).iloc[0].to_dict()
global_best = {
    'score': float(best_row['selection_score']),
    'candidate_dir': Path(best_row['candidate_adapter_dir']),
    'config': {
        'learning_rate': float(best_row['learning_rate']),
        'retain_weight': float(best_row['retain_weight']),
        'kl_weight': float(best_row['kl_weight']),
    },
    'epoch': int(best_row['selected_epoch_for_candidate']),
    'row': best_row,
    'candidate_id': best_row['candidate_id'],
}

if ADAPTER_DIR.exists():
    shutil.rmtree(ADAPTER_DIR)
shutil.copytree(global_best['candidate_dir'], ADAPTER_DIR)
write_json(RESUME_DIR / 'selected_global_best.json', global_best)
print('\nSelected global best:', global_best)
display(candidate_summary_df.head(10))


Existing epoch rows: 69
Completed candidates: ['c01_low_lr_balanced', 'c02_mid_lr_balanced', 'c03_high_lr_balanced', 'c04_lower_retain', 'c05_higher_retain', 'c06_lower_kl', 'c07_higher_kl', 'c08_most_preserving']


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Skipping completed candidate c01_low_lr_balanced
Skipping completed candidate c02_mid_lr_balanced
Skipping completed candidate c03_high_lr_balanced
Skipping completed candidate c04_lower_retain
Skipping completed candidate c05_higher_retain
Skipping completed candidate c06_lower_kl
Skipping completed candidate c07_higher_kl
Skipping completed candidate c08_most_preserving

Candidate: c09_most_aggressive {'learning_rate': 5e-05, 'retain_weight': 1.0, 'kl_weight': 0.1, 'label': 'most_aggressive'}
Resuming c09_most_aggressive from epoch 9 using /content/drive/MyDrive/Thesis/Week 5/results/retain_regularized_unlearning_resumable_v1/resume_state/epoch_checkpoints/c09_most_aggressive/epoch_08/adapter
c09_most_aggressive already has all 8 epochs but was not marked complete. Finalizing from history.


ValueError: Can't find 'adapter_config.json' at '/content/drive/MyDrive/Thesis/Week 5/results/retain_regularized_unlearning_resumable_v1/resume_state/epoch_checkpoints/c09_most_aggressive/epoch_08/adapter'

## 8. Final Evaluation of the Selected Adapter

In [ ]:
del teacher_model, teacher_base
gc.collect()
torch.cuda.empty_cache()

model = load_base_model()
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()
after_forget_df = evaluate(model, eval_forget, 'forget', 'after_retain_regularized_unlearning')
after_retain_df = evaluate(model, eval_retain, 'retain', 'after_retain_regularized_unlearning')
after_general_df = evaluate(model, general_controls, 'general', 'after_retain_regularized_unlearning', general=True, progress_every=0)
after_forget_final_df = evaluate(model, final_forget_excluding_selection, 'forget_final_excluding_selection', 'after_retain_regularized_unlearning')
after_retain_final_df = evaluate(model, final_retain_excluding_selection, 'retain_final_excluding_selection', 'after_retain_regularized_unlearning')

after_forget_df.to_csv(OUTPUT_DIR / 'after_forget_results.csv', index=False)
after_retain_df.to_csv(OUTPUT_DIR / 'after_retain_results.csv', index=False)
after_general_df.to_csv(OUTPUT_DIR / 'after_general_results.csv', index=False)
after_forget_final_df.to_csv(OUTPUT_DIR / 'after_forget_final_excluding_selection_results.csv', index=False)
after_retain_final_df.to_csv(OUTPUT_DIR / 'after_retain_final_excluding_selection_results.csv', index=False)

all_results = pd.concat(
    [before_forget_df, before_retain_df, before_general_df, after_forget_df, after_retain_df, after_general_df],
    ignore_index=True,
)
summary = all_results.groupby(['model_stage', 'eval_split', 'prompt_seen_in_original_training', 'used_for_checkpoint_selection']).agg(
    num_questions=('contains_value', 'size'),
    num_correct=('contains_value', 'sum'),
    contains_value_percentage=('contains_value', lambda values: 100 * values.mean()),
    exact_match_percentage=('exact_match', lambda values: 100 * values.mean()),
).reset_index()

all_results.to_csv(OUTPUT_DIR / 'all_before_after_results.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'percentage_summary.csv', index=False)

display(summary)
print('After forget all:', f'{percentage(after_forget_df):.2f}%')
print('After forget held-out:', f'{prompt_subset_percentage(after_forget_df, False):.2f}%')
print('After retain all:', f'{percentage(after_retain_df):.2f}%')
print('After retain held-out:', f'{prompt_subset_percentage(after_retain_df, False):.2f}%')
print('After general:', f'{percentage(after_general_df):.2f}%')

## 9. Category, Identity, and Week 4 Comparison Outputs

In [ ]:
synthetic_results = pd.concat([before_forget_df, before_retain_df, after_forget_df, after_retain_df], ignore_index=True)
category_summary = synthetic_results.groupby(['model_stage', 'eval_split', 'category']).agg(
    num_questions=('contains_value', 'size'),
    num_correct=('contains_value', 'sum'),
    contains_value_percentage=('contains_value', lambda values: 100 * values.mean()),
    exact_match_percentage=('exact_match', lambda values: 100 * values.mean()),
).reset_index()
identity_summary = synthetic_results.groupby(['model_stage', 'eval_split', 'entity_id']).agg(
    num_questions=('contains_value', 'size'),
    num_correct=('contains_value', 'sum'),
    contains_value_percentage=('contains_value', lambda values: 100 * values.mean()),
).reset_index()
category_summary.to_csv(OUTPUT_DIR / 'category_summary.csv', index=False)
identity_summary.to_csv(OUTPUT_DIR / 'identity_summary.csv', index=False)

metrics = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'run_name': 'week5_retain_regularized_unlearning_resumable_v1',
    'base_model_id': MODEL_ID,
    'source_adapter_dir': str(SOURCE_ADAPTER_DIR),
    'source_week35_metrics_path': str(STRICT_WEEK35_METRICS_PATH),
    'source_week35_run_name': strict_week35_metrics.get('run_name'),
    'source_week35_strict_scoring': strict_week35_metrics.get('strict_scoring'),
    'unlearned_adapter_dir': str(ADAPTER_DIR),
    'method': 'gradient ascent on forget loss plus retain cross-entropy and retain KL regularization against the original Week 3.5 adapter',
    'run_full_grid': RUN_FULL_GRID,
    'num_sweep_candidates': len(SWEEP_CONFIGS),
    'resume_enabled': RESUME_SWEEP,
    'resume_dir': str(RESUME_DIR),
    'selected_candidate_id': global_best['candidate_id'],
    'selected_epoch': int(global_best['epoch']),
    'selected_config': global_best['config'],
    'selection': {
        'forget_selection_examples': len(forget_selection),
        'retain_selection_examples': len(retain_selection),
        'retain_threshold_percentage': MIN_RETAIN_HELDOUT_SELECTION,
        'selection_score': float(global_best['score']),
        'selection_row': global_best['row'],
    },
    'training': {
        'max_epochs': MAX_EPOCHS,
        'batch_size': BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
        'max_grad_norm': MAX_GRAD_NORM,
        'target_forget_max': TARGET_FORGET_MAX,
    },
    'before_unlearning': {
        'forget_all': percentage(before_forget_df),
        'forget_heldout_paraphrases': prompt_subset_percentage(before_forget_df, False),
        'retain_all': percentage(before_retain_df),
        'retain_heldout_paraphrases': prompt_subset_percentage(before_retain_df, False),
        'general': percentage(before_general_df),
    },
    'after_unlearning': {
        'forget_all': percentage(after_forget_df),
        'forget_heldout_paraphrases': prompt_subset_percentage(after_forget_df, False),
        'forget_all_excluding_selection': excluding_selection_percentage(after_forget_df),
        'retain_all': percentage(after_retain_df),
        'retain_heldout_paraphrases': prompt_subset_percentage(after_retain_df, False),
        'retain_all_excluding_selection': excluding_selection_percentage(after_retain_df),
        'general': percentage(after_general_df),
    },
}

(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(ADAPTER_DIR / 'unlearning_config.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')

comparison_rows = []
week4_metrics_path = THESIS_DIR / 'Week 4' / 'results' / 'gradient_ascent_unlearning_v1' / 'results' / 'metrics.json'
if week4_metrics_path.exists():
    week4 = json.loads(week4_metrics_path.read_text(encoding='utf-8'))
    for metric_name in ['forget_all', 'forget_heldout_paraphrases', 'retain_all', 'retain_heldout_paraphrases', 'general']:
        comparison_rows.append({
            'metric': metric_name,
            'week4_after': week4['after_unlearning'].get(metric_name),
            'week5_after': metrics['after_unlearning'].get(metric_name),
            'week5_minus_week4': None if week4['after_unlearning'].get(metric_name) is None else metrics['after_unlearning'].get(metric_name) - week4['after_unlearning'].get(metric_name),
        })
comparison_df = pd.DataFrame(comparison_rows)
if not comparison_df.empty:
    comparison_df.to_csv(OUTPUT_DIR / 'week4_week5_comparison.csv', index=False)
    display(comparison_df)
else:
    print('Week 4 metrics were not found on Drive, so no Week 4 comparison CSV was created.')

print(json.dumps(metrics, indent=2))
print('Saved to:', RUN_DIR)

## Interpretation

A good Week 5 result is not simply the lowest forget accuracy. The main goal is to improve the Week 4 trade-off by preserving retain accuracy, especially held-out retain paraphrases, while still producing a large forget accuracy reduction.

Report both full-evaluation results and the `*_excluding_selection` files. The selection subsets are used only for checkpoint choice, not for parameter updates, but the excluding-selection outputs are cleaner for thesis language.
